1. Load packages, input files, and diagnosis and covariate data. Rename covariates 

2. Define ICD, select only 3 columns (the person ID, phecode, and diagnosis date), define cases (and/or for hearing loss and tinnitus, rule of 2), define controls (anyone who is not a case), filter unrelated samples, make table with only the controls and cases (which includes everyone because the control definition is very liberal) 

In [1]:
# Step 1: Load packages
import pandas as pd
import numpy as np

# Step 2: Load input files
base_path = "/static/PMBB/PMBB-Release-2026-4.0/Phenotype/4.0/"
COV_file = base_path + "PMBB-Release-2026-4.0_phenotype_covariates.txt"
DX_file = base_path + "PMBB-Release-2026-4.0_phenotype_condition_occurrence.txt"
PROC_file = base_path + "PMBB-Release-2026-4.0_phenotype_procedure_occurrence.txt"

# Don't need to get rid of unrelated because SAIGE can account for relationships 
# UNREL_FILE = "/static/PMBB/PMBB-Release-2026-4.0/Exome/relationships/PMBB-Release-2026-4.0_genetic_exome.2nd_degree_unrelated.txt"

# Step 3: Load diagnosis and covariate data
cols = pd.read_csv(DX_file, sep="\t", nrows=5)
DX_v4 = pd.read_csv(
    DX_file,
    sep="\t",
    usecols=[
        "person_id",
        "condition_source_value",
        "condition_start_date"
    ]
)
print(DX_v4.columns)
covars = pd.read_csv(COV_file, sep="\t")
print(covars.columns)

# Step 4: Recode and rename covariates
covars.columns = covars.columns.str.strip()  # prevents hidden whitespace bugs

covars['SEX'] = covars['sequenced_gender'].replace({'Male': 1, 'Female': 2})

phenos = covars[
    ["person_id", "batch", "sample_age", "SEX"]
].copy()

phenos = phenos.rename(columns={
    "person_id": "PMBB_ID",
    "batch": "Batch",
    "sample_age": "AGE"
})

phenos["PMBB_ID"] = phenos["PMBB_ID"].astype(str)

HL_ICD = ["H90", "H91"]
TIN_ICD = ["H93"]

# -----------------------------
# 0) Harmonize ID types
# -----------------------------
DX = DX_v4[['person_id', 'condition_source_value', 'condition_start_date']].copy()
print(DX["condition_source_value"].head(20))
DX['person_id'] = DX['person_id'].astype(str)

# -----------------------------
# 1) Define HL and tinnitus cases (RULE OF 2)
# -----------------------------
hl_dx = DX[
    DX["condition_source_value"]
      .astype(str)
      .str.contains("|".join(HL_ICD), regex=True, na=False)
]
hl_counts = (
    hl_dx.dropna(subset=['condition_start_date'])
        .groupby('person_id')['condition_start_date']
        .nunique()
        .reset_index()
        .rename(columns={
            'person_id': 'PMBB_ID',
            'condition_start_date': 'HL_count'
        })
)

hl_counts['HL_case'] = (hl_counts['HL_count'] >= 2).astype(int)

phenos = phenos.merge(
    hl_counts[['PMBB_ID', 'HL_case', 'HL_count']],
    how='left',
    on='PMBB_ID'
)
phenos['HL_case'] = phenos['HL_case'].fillna(0).astype(int)
phenos['HL_count'] = phenos['HL_count'].fillna(0).astype(int)

tin_dx = DX[
    DX["condition_source_value"]
      .astype(str)
      .str.contains("|".join(TIN_ICD), regex=True, na=False)
]
tin_counts = (
    tin_dx.dropna(subset=['condition_start_date'])
        .groupby('person_id')['condition_start_date']
        .nunique()
        .reset_index()
        .rename(columns={
            'person_id': 'PMBB_ID',
            'condition_start_date': 'TIN_count'
        })
)

tin_counts['TIN_case'] = (tin_counts['TIN_count'] >= 2).astype(int)

print(phenos.columns)
print(hl_counts.columns)

phenos = phenos.merge(
    tin_counts[['PMBB_ID', 'TIN_case', 'TIN_count']],
    how='left',
    on='PMBB_ID'
)
phenos['TIN_case'] = phenos['TIN_case'].fillna(0).astype(int)
phenos['TIN_count'] = phenos['TIN_count'].fillna(0).astype(int)

# -----------------------------
# 2) Filter to unrelated; NOT NEEDED IF USING SAIGE 
# -----------------------------
#unrel_ids = pd.read_csv(UNREL_FILE, header=None)[0].astype(str).unique()
#phenos = phenos[phenos['PMBB_ID'].isin(unrel_ids)].copy()

# -----------------------------
# 3) Define case and control
# -----------------------------
phenos['CASE'] = ((phenos['HL_case'] == 1) | (phenos['TIN_case'] == 1)).astype(int)

phenos['CONTROL'] = np.where(
    (phenos['CASE'] == 0),
    1, 
    0
)

# -----------------------------
# 4) Final binary case-control dataset
#    Keep ONLY cases and  controls
# -----------------------------
phenos_cc = phenos[
    (phenos['CASE'] == 1) | (phenos['CONTROL'] == 1)
].copy()
# Keeps everyone who is a case or a control (which is basically everyone)

# Final phenotype label: 1=case, 0=control
phenos_cc['PHENO'] = phenos_cc['CASE']

# -----------------------------
# 5) Summary
# -----------------------------

print("Hearing loss + tinnitus ExWAS phenotype")
print("Total:", len(phenos_cc))
print("Cases:", phenos_cc['PHENO'].sum())
print("Controls:", (phenos_cc['PHENO'] == 0).sum())

# -----------------------------
# 6) Save outputs
# -----------------------------
outpath = "/project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS"

phenos_cc[['PMBB_ID', 'Batch', 'AGE', 'SEX', 'PHENO']].to_csv(
    f"{outpath}.csv", index=False
)

phenos_cc[['PMBB_ID', 'Batch', 'AGE', 'SEX', 'PHENO']].to_csv(
    f"{outpath}.txt", sep='\t', index=False
)


Index(['person_id', 'condition_start_date', 'condition_source_value'], dtype='str')
Index(['person_id', 'batch', 'crep_highrisk_flag', 'sequenced_gender',
       'sample_date', 'sample_age', 'exome_PC1', 'exome_PC2', 'exome_PC3',
       'exome_PC4', 'exome_PC5', 'exome_PC6'],
      dtype='str')
0     528.6
1     528.6
2     528.6
3     729.5
4     729.5
5     729.5
6     729.5
7     729.5
8     729.5
9     729.5
10    729.5
11    729.5
12    729.5
13    729.5
14    729.5
15    729.5
16    729.5
17    729.5
18    729.5
19    729.5
Name: condition_source_value, dtype: str
Index(['PMBB_ID', 'Batch', 'AGE', 'SEX', 'HL_case', 'HL_count'], dtype='str')
Index(['PMBB_ID', 'HL_count', 'HL_case'], dtype='str')
Hearing loss + tinnitus ExWAS phenotype
Total: 70925
Cases: 6086
Controls: 64839


2. VERSION 2: Look at both procedure codes and ICD. Didn't work because kernel kept dying :(((

In [1]:
# Step 1: Load packages
import pandas as pd
import numpy as np

# Step 2: Load input files
base_path = "/static/PMBB/PMBB-Release-2026-4.0/Phenotype/4.0/"
COV_file = base_path + "PMBB-Release-2026-4.0_phenotype_covariates.txt"
DX_file = base_path + "PMBB-Release-2026-4.0_phenotype_condition_occurrence.txt"
PROC_file = base_path + "PMBB-Release-2026-4.0_phenotype_procedure_occurrence.txt"

# Don't need to get rid of unrelated because SAIGE can account for relationships 
# UNREL_FILE = "/static/PMBB/PMBB-Release-2026-4.0/Exome/relationships/PMBB-Release-2026-4.0_genetic_exome.2nd_degree_unrelated.txt"

# Step 3: Load diagnosis and covariate data
DX_v4 = pd.read_csv(
    DX_file,
    sep="\t",
    usecols=[
        "person_id",
        "condition_source_value",
        "condition_start_date"
    ]
)
print(DX_v4.columns)
covars = pd.read_csv(COV_file, sep="\t")
print(covars.columns)

# Step 4: Recode and rename covariates
covars.columns = covars.columns.str.strip()  # prevents hidden whitespace bugs

covars['SEX'] = covars['sequenced_gender'].replace({'Male': 1, 'Female': 2})

phenos = covars[
    ["person_id", "batch", "sample_age", "SEX"]
].copy()

phenos = phenos.rename(columns={
    "person_id": "PMBB_ID",
    "batch": "Batch",
    "sample_age": "AGE"
})

phenos["PMBB_ID"] = phenos["PMBB_ID"].astype(str)

HL_ICD = ["H90", "H91"]
TIN_ICD = ["H93"]

# -----------------------------
# 0) Harmonize ID types
# -----------------------------
DX = DX_v4[['person_id', 'condition_source_value', 'condition_start_date']].copy()
print(DX["condition_source_value"].head(20))
DX['person_id'] = DX['person_id'].astype(str)

# -----------------------------
# 1) Define HL and tinnitus cases (RULE OF 2)
# -----------------------------
hl_dx = DX[
    DX["condition_source_value"]
      .astype(str)
      .str.contains("|".join(HL_ICD), regex=True, na=False)
]
hl_counts = (
    hl_dx.dropna(subset=['condition_start_date'])
        .groupby('person_id')['condition_start_date']
        .nunique()
        .reset_index()
        .rename(columns={
            'person_id': 'PMBB_ID',
            'condition_start_date': 'HL_count'
        })
)

hl_counts['HL_case'] = (hl_counts['HL_count'] >= 2).astype(int)

phenos = phenos.merge(
    hl_counts[['PMBB_ID', 'HL_case', 'HL_count']],
    how='left',
    on='PMBB_ID'
)
phenos['HL_case'] = phenos['HL_case'].fillna(0).astype(int)
phenos['HL_count'] = phenos['HL_count'].fillna(0).astype(int)

tin_dx = DX[
    DX["condition_source_value"]
      .astype(str)
      .str.contains("|".join(TIN_ICD), regex=True, na=False)
]
tin_counts = (
    tin_dx.dropna(subset=['condition_start_date'])
        .groupby('person_id')['condition_start_date']
        .nunique()
        .reset_index()
        .rename(columns={
            'person_id': 'PMBB_ID',
            'condition_start_date': 'TIN_count'
        })
)

tin_counts['TIN_case'] = (tin_counts['TIN_count'] >= 2).astype(int)

print(phenos.columns)
print(hl_counts.columns)

phenos = phenos.merge(
    tin_counts[['PMBB_ID', 'TIN_case', 'TIN_count']],
    how='left',
    on='PMBB_ID'
)
phenos['TIN_case'] = phenos['TIN_case'].fillna(0).astype(int)
phenos['TIN_count'] = phenos['TIN_count'].fillna(0).astype(int)


del DX_v4
del DX
del hl_dx
del tin_dx

# OMOP procedure stuff 

keywords = [
    "HEARING",
    "AUDIO",
    "COCHLEAR"
]

pattern = "|".join(keywords)

proc_ids = set()

for chunk in pd.read_csv(
    PROC_file,
    sep="\t",
    usecols=[
        "person_id",
        "procedure_source_value"
    ],
    chunksize=100000,
):

    mask = (
        chunk["procedure_source_value"]
        .astype(str)
        .str.upper()
        .str.contains(pattern, regex=True, na=False)
    )

    proc_ids.update(
        chunk.loc[mask, "person_id"].astype(str)
    )

    del chunk


proc_hl_ids = pd.DataFrame({
    "PMBB_ID": list(proc_ids),
    "PROC_HL": 1
})

print("Procedure-defined HL participants:", len(proc_hl_ids))


# -----------------------------
# 2) Filter to unrelated; NOT NEEDED IF USING SAIGE 
# -----------------------------
#unrel_ids = pd.read_csv(UNREL_FILE, header=None)[0].astype(str).unique()
#phenos = phenos[phenos['PMBB_ID'].isin(unrel_ids)].copy()

# -----------------------------
# 3) Define case and control
# -----------------------------
# Merge procedure-defined hearing cases
phenos = phenos.merge(
    proc_hl_ids,
    on="PMBB_ID",
    how="left"
)

phenos["PROC_HL"] = phenos["PROC_HL"].fillna(0).astype(int)


phenos['CASE'] = (
    (phenos['HL_case'] == 1) |
    (phenos['TIN_case'] == 1) |
    (phenos['PROC_HL'] == 1)
).astype(int)

phenos['CONTROL'] = np.where(
    (phenos['CASE'] == 0),
    1, 
    0
)

# -----------------------------
# 4) Final binary case-control dataset
#    Keep ONLY cases and  controls
# -----------------------------
phenos_cc = phenos[
    (phenos['CASE'] == 1) | (phenos['CONTROL'] == 1)
].copy()
# Keeps everyone who is a case or a control (which is basically everyone)

# Final phenotype label: 1=case, 0=control
phenos_cc['PHENO'] = phenos_cc['CASE']

# -----------------------------
# 5) Summary
# -----------------------------

print("Hearing loss + tinnitus ExWAS phenotype")
print("Total:", len(phenos_cc))
print("Diagnosis HL cases:", phenos["HL_case"].sum())
print("Tinnitus cases:", phenos["TIN_case"].sum())
print("Procedure cases:", phenos["PROC_HL"].sum())
print("Final cases:", phenos["CASE"].sum())

print("Cases:", phenos_cc['PHENO'].sum())
print("Controls:", (phenos_cc['PHENO'] == 0).sum())

# -----------------------------
# 6) Save outputs
# -----------------------------
outpath = "/project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS/phenotypingQC_withOMOP"

phenos_cc[['PMBB_ID', 'Batch', 'AGE', 'SEX', 'PHENO']].to_csv(
    f"{outpath}.csv", index=False
)

phenos_cc[['PMBB_ID', 'Batch', 'AGE', 'SEX', 'PHENO']].to_csv(
    f"{outpath}.txt", sep='\t', index=False
)


Index(['person_id', 'condition_start_date', 'condition_source_value'], dtype='str')
Index(['person_id', 'batch', 'crep_highrisk_flag', 'sequenced_gender',
       'sample_date', 'sample_age', 'exome_PC1', 'exome_PC2', 'exome_PC3',
       'exome_PC4', 'exome_PC5', 'exome_PC6'],
      dtype='str')
0     528.6
1     528.6
2     528.6
3     729.5
4     729.5
5     729.5
6     729.5
7     729.5
8     729.5
9     729.5
10    729.5
11    729.5
12    729.5
13    729.5
14    729.5
15    729.5
16    729.5
17    729.5
18    729.5
19    729.5
Name: condition_source_value, dtype: str


: 

3. Creates a file for PLINK (uses FID and IID instead of the PMBB IDs)

In [14]:
import pandas as pd
import numpy as np

###Make keep file for plink###
# Load final phenotype table
pheno_file = "/project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS.csv"
phenos = pd.read_csv(pheno_file)


# Extract FID and IID (both = PMBB_ID)
keep_df = phenos[['PMBB_ID']].copy()
keep_df['FID'] = keep_df['PMBB_ID']
keep_df['IID'] = keep_df['PMBB_ID']


# Reorder columns
keep_df = keep_df[['FID', 'IID']]


# Save as keep file
keep_file = "/project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS/HL_TIN_PMBBv4_keep.txt"
# Load phenotype/keep file
phenotype_df = pd.read_csv(
    keep_file,
    sep="\t",
    header=None
)

phenotype_df.columns = ["FID", "IID"]

print(f"PLINK keep file written to: {keep_file}")

PLINK keep file written to: /project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS/HL_TIN_PMBBv4_keep.txt


4. QC check of PMBB data, checks how many of the samples actually have genotype data, makes new file with that data

In [16]:
import pandas as pd

# File paths
keep_path = "/project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS/HL_TIN_PMBBv4_keep.txt"
fam_path = "/static/PMBB/PMBB-Release-2026-4.0/Imputed/common_snps_LD_pruned/PMBB-Release-2026-4.0_genetic_imputed.commonsnps.ldpruned.ALL.fam"
output_path = "/project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS/pheno_PMBBv4_final_samplelist_filtered.txt"

# Load FAM
fam_df = pd.read_csv(
    fam_path,
    sep=r"\s+",
    header=None
)

fam_df.columns = ["FID", "IID", "Father", "Mother", "Sex", "Phenotype"]

fam_df["IID"] = fam_df["IID"].astype(str)


# Load keep file
phenotype_df = pd.read_csv(
    keep_path,
    sep="\t",
    header=None
)

phenotype_df.columns = ["FID", "IID"]

phenotype_df["IID"] = phenotype_df["IID"].astype(str)


print("FAM samples:", len(fam_df))
print("Phenotype samples:", len(phenotype_df))


# Keep only samples with genotype data
filtered_df = phenotype_df.merge(
    fam_df[["IID"]],
    on="IID",
    how="inner"
)


print("Matched samples:", len(filtered_df))


# Save
filtered_df.to_csv(
    output_path,
    sep="\t",
    index=False
)

print(f"Saved filtered file to: {output_path}")

FAM samples: 70493
Phenotype samples: 70925
Matched samples: 70408
Saved filtered file to: /project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS/pheno_PMBBv4_final_samplelist_filtered.txt


5. Checks how many of the HL and tinnitus samples actually have genotype data and makes a new file with that

In [17]:
import pandas as pd

# Paths
pheno_file = "/project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS.txt"
fam_file = "/static/PMBB/PMBB-Release-2026-4.0/Imputed/common_snps_LD_pruned/PMBB-Release-2026-4.0_genetic_imputed.commonsnps.ldpruned.ALL.fam"
output_file = "/project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS/HL_TIN_PMBBv4_SAIGE_samples.txt"

pheno = pd.read_csv(
    pheno_file,
    sep="\t"
)

# Make sure ID is string
pheno["PMBB_ID"] = pheno["PMBB_ID"].astype(str)

# Load genotype sample list (.fam)

fam = pd.read_csv(
    fam_file,
    sep=r"\s+",
    header=None
)

fam.columns = [
    "FID",
    "IID",
    "Father",
    "Mother",
    "Sex",
    "Phenotype"
]

fam["IID"] = fam["IID"].astype(str)

# Match phenotype samples to genotype samples

matched = pheno[
    pheno["PMBB_ID"].isin(fam["IID"])
].copy()


# Create SAIGE phenotype file

saige_pheno = matched[
    [
        "PMBB_ID",
        "PHENO",
        "AGE",
        "SEX",
        "Batch"
    ]
].copy()

saige_pheno = saige_pheno.rename(
    columns={"PMBB_ID": "IID"}
)

# Save

saige_pheno.to_csv(
    output_file,
    sep="\t",
    index=False
)


# Summary

print("SAIGE sample file created")
print("Saved:", output_file)
print()

print("Total genotype + phenotype samples:", len(saige_pheno))
print("Cases:", (saige_pheno["PHENO"] == 1).sum())
print("Controls:", (saige_pheno["PHENO"] == 0).sum())

SAIGE sample file created
Saved: /project/hall/analysis/hearing-loss-genomics/analysis/elena/rarevariantExWAS/HL_TIN_PMBBv4_SAIGE_samples.txt

Total genotype + phenotype samples: 70408
Cases: 6050
Controls: 64358
